# Thesis table check

Every table in the PDF is rebuilt here from the result CSVs in `experiments/`. Each section has the same layout:

1. **What the table shows** and **how each number is computed**, in plain words
2. **Files** it reads (paths relative to the repo root)
3. a **worked example** on one model / dataset that shows the raw rows, so you can see where a number comes from
4. the **full table**, in the same row order, column order and rounding as the PDF, for side-by-side comparison

Words used throughout:
- **span s→e**: blocks s+1 … e are replaced by one linear translator (e.g. 3→4 replaces block 4 only).
- **drop** (pp) = original score − score after the approximation. Positive = worse, negative = better than the original.
  Score = accuracy on natural images, AUROC on medical. All scores are means over 3 seeds.
- **Spearman ρ**: rank correlation. +1 = the two lists are in exactly the same order, 0 = unrelated, −1 = reversed.

Run from this folder or the repo root: Restart & Run All.

All tables reproduce the PDF to the printed decimals, except for a few Table A.2 (DeiT-S) and Table C.8 cells
that differ by at most 0.08 pp and 0.01 respectively; see those sections.


In [1]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

# ---- where the result files are (the experiments/ folders of this repo) ----
EXP = Path.cwd().resolve()
while not (EXP / "experiments").is_dir():  # works from the repo root or any folder below it
    EXP = EXP.parent
EXP = EXP / "experiments"
S2 = EXP / "4_1-4_2_choosing_blocks" / "natural"  # natural images, single spans (Sec. 4.1)
S3 = EXP / "4_3_combining"                      # natural images, two-block approximations (Sec. 4.3)
S4 = EXP / "4_1-4_2_choosing_blocks" / "medical"  # medical images, single spans (Sec. 4.1, 4.2)
S5 = EXP / "4_4_without_target_data"            # translator fitted on another dataset (Sec. 4.4)

# ---- names ----
MODEL_NAME = {"deitsmall": "DeiT-S", "vitsmall": "ViT-S", "dinov2base": "DINOv2-B", "vitlarge": "ViT-L",
              "raddino": "RAD-DINO", "raddinomaira2": "RAD-DINO-MAIRA-2", "phikonv2": "Phikon-v2"}
DATASET_NAME = {"cifar100": "C100", "imagenet1k": "IN-1k", "chestxray14multilabel": "CXR14",
                "chestxraypneumonia": "Pneum.", "patchcamelyon": "PCam", "tcgaut": "TCGA-UT"}

# ---- the 18 model / dataset combinations ----
NATURAL = [("deitsmall", "cifar100"), ("deitsmall", "imagenet1k"), ("vitsmall", "cifar100"), ("vitsmall", "imagenet1k"),
           ("dinov2base", "cifar100"), ("dinov2base", "imagenet1k"), ("vitlarge", "cifar100"), ("vitlarge", "imagenet1k")]
DOMAIN_SPECIFIC = [("raddino", "chestxray14multilabel"), ("raddino", "chestxraypneumonia"),
                   ("raddinomaira2", "chestxray14multilabel"), ("raddinomaira2", "chestxraypneumonia"),
                   ("phikonv2", "patchcamelyon"), ("phikonv2", "tcgaut")]
GENERAL_PURPOSE = [("dinov2base", "chestxray14multilabel"), ("dinov2base", "chestxraypneumonia"),
                   ("vitlarge", "patchcamelyon"), ("vitlarge", "tcgaut")]
MEDICAL = DOMAIN_SPECIFIC + GENERAL_PURPOSE
GROUPS = {"Natural": NATURAL, "Domain-specific": DOMAIN_SPECIFIC, "General-purpose": GENERAL_PURPOSE}


def group_of(model, dataset):
    for group, combos in GROUPS.items():
        if (model, dataset) in combos:
            return group


def label(model, dataset):
    return f"{MODEL_NAME[model]} / {DATASET_NAME[dataset]}"


def f2(x):
    # Format like the PDF: 2 decimals, '–' for missing.
    return "–" if pd.isna(x) else f"{x:.2f}"


def spearman(a, b):
    return spearmanr(a, b)[0]

In [2]:
# ---------------- loading helpers (used by most tables) ----------------

def parse_spans(text):
    # '[(2, 3)]' -> [(2, 3)];  '[(0, 1), (4, 5)]' -> two spans;  '[]' -> the original, unapproximated model
    return [tuple(s) for s in ast.literal_eval(text)]


def load_drops(model, dataset):
    # Returns (original score in %, one row per span s->e with its drop and seed SD in pp).
    # The accuracy files hold one row per approximation, already averaged over the 3 seeds.
    if (model, dataset) in NATURAL:
        acc = pd.read_csv(S2 / "accuracies" / f"accuracies_results_{model}_{dataset}.csv")
        score = "accuracy"
    else:
        acc = pd.read_csv(S4 / "accuracies" / "combined" / f"accuracies_results_{model}_{dataset}.csv")
        score = "auroc"
    original = acc.loc[acc["approx_layer"] == "[]", f"{score}_mean"].iloc[0] * 100

    rows = []
    for _, r in acc.iterrows():
        spans = parse_spans(r["approx_layer"])
        if len(spans) != 1:  # the original model has no span
            continue
        start, end = spans[0]
        rows.append({"start": start, "end": end, "length": end - start,
                     "drop": original - r[f"{score}_mean"] * 100,
                     "seed_sd": r[f"{score}_std"] * 100})
    return original, pd.DataFrame(rows)


def load_metrics(model, dataset):
    # Similarity metrics of every span s->e, computed on the 500 calibration images (one row per span).
    if (model, dataset) in NATURAL:
        path = S2 / "analysis" / f"output_level_eval_s2_{model}_{dataset}.csv"
    else:
        path = S4 / "analysis" / f"output_level_eval_s4_{model}_{dataset}.csv"
    return pd.read_csv(path).rename(columns={"skip_from": "start", "skip_to": "end"})


def load_spans(model, dataset):
    # Metrics and true drop side by side, one row per span (keeps the metric file's row order; only matters for ties).
    original, drops = load_drops(model, dataset)
    return load_metrics(model, dataset).merge(drops, on=["start", "end"])


# Quick look at what one combination looks like:
load_spans("deitsmall", "cifar100")[["start", "end", "length", "drop", "angular_dist", "cosine_sim", "local_cka",
                                     "cka_embed", "toast_linear_error_alltok"]].head()

,start,end,length,drop,angular_dist,cosine_sim,local_cka,cka_embed,toast_linear_error_alltok
0,0,1,1,9.273333,0.228805,0.808320,0.845710,0.734296,0.309085
1,0,2,2,22.783333,0.330593,0.623260,0.705916,0.537737,0.497655
2,0,3,3,23.423333,0.369143,0.617839,0.482692,0.521706,0.617175
3,0,4,4,25.500000,0.392422,0.604054,0.207613,0.484539,0.706564
4,1,2,1,6.956667,0.221036,0.863873,0.891669,0.798934,0.367840


---
# Section 4.1: ranking single spans (Tables 1, C.1, C.2, C.3)

**The question:** if we sort all candidate spans by a similarity metric, does that order match the true drops?

For one model / dataset (38 spans for a 12-block model, 86 for a 24-block one), and one metric:

1. Turn the metric into a **safety score** (higher = safer). Distances and errors (angular distance, TOAST error, KL)
   are flipped with a minus sign, since a big distance means the span changes a lot. Similarities (CKA, cosine) stay as they are.
2. **ρ** = Spearman(safety score, −drop) over all spans. Positive = the metric points the right way.
3. **ρ_len** = the same ρ but computed separately among spans of length 1, 2, 3 and 4, then averaged.
   This removes the easy part ("long spans are worse") and asks: for a fixed number of blocks, does it pick the right ones?
4. **Top-5 drop** = average true drop of the 5 spans with the highest safety score.

Baselines: **Fewest blocks** ranks by span length only (ρ = Spearman(−length, −drop); its top 5 are random single
blocks, so its top-5 drop = the mean drop of a single block). **Oracle** = mean drop of the 5 truly best spans.

**Files:** `experiments/4_1-4_2_choosing_blocks/natural/accuracies/` + `.../analysis/output_level_eval_s2_*.csv` (natural),
`experiments/4_1-4_2_choosing_blocks/medical/accuracies/combined/` + `.../analysis/output_level_eval_s4_*.csv` (medical).
**Original code:** `experiments/4_1-4_2_choosing_blocks/metric_table.ipynb`, cells 1–2.

In [3]:
# Raw direction of each metric. Distance/error: big = span changes a lot = RISKY. Similarity: big = SAFE.
BIG_IS_RISKY = {"angular_dist", "toast_linear_error", "toast_linear_error_alltok", "kl_div"}

METRIC_NAME = {"angular_dist": "Angular distance",
               "local_cka": "Local CKA",
               "toast_linear_error_alltok": "Linear approx. error (TOAST)",  # fitted on all tokens
               "cka_embed": "Global CKA",
               "cosine_sim": "Cosine similarity"}
EXTRA_METRICS = ["toast_linear_error",   # TOAST error fitted on CLS tokens only (Table C.1)
                 "orig_head_accuracy",   # pretrained-head accuracy, ImageNet-1k head models only (Table C.2)
                 "kl_div"]               # KL divergence, same 3 combinations (Table C.2)


def safety_score(spans, metric):
    return -spans[metric] if metric in BIG_IS_RISKY else spans[metric]


def rank_quality(spans, metric):
    # Returns rho, rho_len, top-5 drop of one metric on one model / dataset (steps 1-4 above).
    safety = safety_score(spans, metric)
    goodness = -spans["drop"]  # higher = better approximation

    rho = spearman(safety, goodness)

    rho_per_length = []
    for length in [1, 2, 3, 4]:
        same = spans["length"] == length
        rho_per_length.append(spearman(safety[same], goodness[same]))
    rho_len = np.mean(rho_per_length)

    top5_spans = safety.nlargest(5).index
    top5_drop = spans.loc[top5_spans, "drop"].mean()
    return rho, rho_len, top5_drop

### Worked example: DeiT-S / CIFAR-100, angular distance

The 38 spans sorted from safest to riskiest according to angular distance. The metric "picks" the first 5 rows.

In [4]:
spans = load_spans("deitsmall", "cifar100")
example = spans[["start", "end", "length", "angular_dist", "drop"]].copy()
example["safety (= -angular_dist)"] = -example["angular_dist"]
example = example.sort_values("safety (= -angular_dist)", ascending=False)
display(example.round(3))

print("Top 5 picks:", [f"{s}->{e}" for s, e in zip(example.start[:5], example.end[:5])])
print("Their drops:", example["drop"][:5].round(2).tolist(), "-> mean =", round(example["drop"][:5].mean(), 2), "pp  (Table C.2b: 0.17)")

for length in [1, 2, 3, 4]:
    same = example[example.length == length]
    print(f"length {length}: {len(same)} spans, rho = {spearman(-same.angular_dist, -same['drop']):.2f}")
rho, rho_len, top5 = rank_quality(spans, "angular_dist")
print(f"rho over all spans = {rho:.2f};  rho_len = mean of the 4 above = {rho_len:.2f}  (Table C.2a: 0.93)")

,start,end,length,angular_dist,drop,safety (= -angular_dist)
35,9,10,1,0.081,-0.307,-0.081
37,10,11,1,0.105,-0.533,-0.105
32,8,9,1,0.117,0.537,-0.117
28,7,8,1,0.130,0.487,-0.130
24,6,7,1,0.142,0.680,-0.142
36,9,11,2,0.142,-0.463,-0.142
33,8,10,2,0.149,0.173,-0.149
20,5,6,1,0.161,2.470,-0.161
16,4,5,1,0.170,1.677,-0.170
12,3,4,1,0.181,1.030,-0.181


Top 5 picks: ['9->10', '10->11', '8->9', '7->8', '6->7']
Their drops: [-0.31, -0.53, 0.54, 0.49, 0.68] -> mean = 0.17 pp  (Table C.2b: 0.17)
length 1: 11 spans, rho = 0.90
length 2: 10 spans, rho = 0.89
length 3: 9 spans, rho = 0.93
length 4: 8 spans, rho = 0.98
rho over all spans = 0.96;  rho_len = mean of the 4 above = 0.93  (Table C.2a: 0.93)


### Compute every metric on every combination

`R` has one row per (model, dataset, metric). Tables 1, C.1 and C.2 are just averages / slices of it.

In [5]:
rows = []
for model, dataset in NATURAL + MEDICAL:
    spans = load_spans(model, dataset)
    base = {"model": model, "dataset": dataset, "group": group_of(model, dataset), "n_spans": len(spans)}

    for metric in list(METRIC_NAME) + EXTRA_METRICS:
        if spans[metric].isna().all():  # head metrics only exist for 3 ImageNet-1k combinations
            continue
        rho, rho_len, top5 = rank_quality(spans, metric)
        rows.append({**base, "metric": metric, "rho": rho, "rho_len": rho_len, "top5": top5})

    single_blocks = spans[spans["length"] == 1]
    rows.append({**base, "metric": "fewest", "rho": spearman(-spans["length"], -spans["drop"]),
                 "top5": single_blocks["drop"].mean()})
    rows.append({**base, "metric": "oracle", "top5": spans["drop"].nsmallest(5).mean()})
    # seed noise of a top-5 mean (Table C.2b, SD column): average one-span SD / sqrt(5)
    rows.append({**base, "metric": "seed_sd_top5", "top5": spans["seed_sd"].mean() / np.sqrt(5)})

R = pd.DataFrame(rows)


def get(model, dataset, metric, column):
    # One number from R (NaN if that metric does not exist for that combination).
    sel = R[(R.model == model) & (R.dataset == dataset) & (R.metric == metric)]
    return sel[column].iloc[0] if len(sel) else np.nan


def group_mean(group, metric, column):
    sel = R[(R.group == group) & (R.metric == metric)]
    return sel[column].mean()


print("number of spans per combination:")
print(R.drop_duplicates(["model", "dataset"]).groupby("n_spans").size().to_string())

number of spans per combination:
n_spans
38    12
86     6


## Table 1 — Ranking approximations by similarity (p. 4)

Each cell = mean over the combinations of that group (8 natural, 6 domain-specific, 4 general-purpose).
Number in brackets = in how many combinations the metric has the highest ρ_len of the five (ties at 2 decimals count for each).

In [6]:
# Count wins: for each combination, which metric(s) have the highest rho_len (rounded to 2 decimals)?
wins = {(group, metric): 0 for group in GROUPS for metric in METRIC_NAME}
for model, dataset in NATURAL + MEDICAL:
    rho_lens = {metric: round(get(model, dataset, metric, "rho_len"), 2) for metric in METRIC_NAME}
    best = max(rho_lens.values())
    for metric, value in rho_lens.items():
        if value == best:
            wins[(group_of(model, dataset), metric)] += 1

table = []
for metric, nice_name in METRIC_NAME.items():
    table.append({
        "Metric": nice_name,
        "Nat ρ": f2(group_mean("Natural", metric, "rho")),
        "Nat ρlen (wins)": f"{f2(group_mean('Natural', metric, 'rho_len'))} ({wins[('Natural', metric)]})",
        "Nat Top-5 drop": f2(group_mean("Natural", metric, "top5")),
        "Dom-spec ρ": f2(group_mean("Domain-specific", metric, "rho")),
        "Dom-spec ρlen (wins)": f"{f2(group_mean('Domain-specific', metric, 'rho_len'))} ({wins[('Domain-specific', metric)]})",
        "Gen-purp ρ": f2(group_mean("General-purpose", metric, "rho")),
        "Gen-purp ρlen (wins)": f"{f2(group_mean('General-purpose', metric, 'rho_len'))} ({wins[('General-purpose', metric)]})",
    })
table.append({"Metric": "Fewest blocks (random single block)",
              "Nat ρ": f2(group_mean("Natural", "fewest", "rho")), "Nat Top-5 drop": f2(group_mean("Natural", "fewest", "top5")),
              "Dom-spec ρ": f2(group_mean("Domain-specific", "fewest", "rho")),
              "Gen-purp ρ": f2(group_mean("General-purpose", "fewest", "rho"))})
table.append({"Metric": "Oracle", "Nat Top-5 drop": f2(group_mean("Natural", "oracle", "top5"))})
display(pd.DataFrame(table).set_index("Metric").fillna("–"))

# Text check (Sec. 4.1 and C.2 caption): "a random single block costs 0.06 pp AUROC on average (at most 0.51 pp)"
medical_single = R[(R.metric == "fewest") & (R.group != "Natural")]["top5"]
print(f"medical, mean drop of a single block: average over 10 combinations = {medical_single.mean():.2f} pp, "
      f"largest combination = {medical_single.max():.2f} pp")

,Nat ρ,Nat ρlen (wins),Nat Top-5 drop,Dom-spec ρ,Dom-spec ρlen (wins),Gen-purp ρ,Gen-purp ρlen (wins)
Metric,,,,,,,
Angular distance,0.85,0.50 (1),1.75,0.48,0.44 (2),0.43,0.23 (0)
Local CKA,0.66,0.23 (0),4.05,0.38,0.19 (0),0.61,0.51 (2)
Linear approx. error (TOAST),0.75,0.35 (1),4.57,0.50,0.49 (3),0.38,0.20 (0)
Global CKA,0.91,0.71 (2),1.66,0.20,-0.21 (0),0.54,0.46 (0)
Cosine similarity,0.91,0.73 (5),1.53,0.17,-0.25 (1),0.54,0.49 (2)
Fewest blocks (random single block),0.81,–,4.00,0.36,–,0.25,–
Oracle,–,–,1.14,–,–,–,–


medical, mean drop of a single block: average over 10 combinations = 0.06 pp, largest combination = 0.51 pp


## Table C.1 — TOAST error fitted on CLS tokens only vs all tokens (p. 17)

Same computation as Table 1, for the two versions of the TOAST error. The "All tokens" row should equal Table 1's TOAST row.

In [7]:
table = []
for metric, nice_name in [("toast_linear_error", "CLS tokens only"), ("toast_linear_error_alltok", "All tokens")]:
    table.append({"Fitted on": nice_name,
                  "Nat ρlen": f2(group_mean("Natural", metric, "rho_len")),
                  "Nat Top-5 drop": f2(group_mean("Natural", metric, "top5")),
                  "Dom-spec ρlen": f2(group_mean("Domain-specific", metric, "rho_len")),
                  "Gen-purp ρlen": f2(group_mean("General-purpose", metric, "rho_len"))})
pd.DataFrame(table).set_index("Fitted on")

,Nat ρlen,Nat Top-5 drop,Dom-spec ρlen,Gen-purp ρlen
Fitted on,,,,
CLS tokens only,0.00,4.51,0.22,0.18
All tokens,0.35,4.57,0.49,0.20


## Table C.2 — Metric quality per model and dataset (p. 18)

The per-combination numbers behind Table 1. (a) ρ_len of each metric, plus the all-span ρ of Fewest blocks.
(b) Top-5 drop on natural images, plus Fewest, Oracle and SD (seed noise of a top-5 mean).
The Head and KL means are over their 3 rows only.

In [8]:
SHORT = {"angular_dist": "Ang.", "local_cka": "L-CKA", "toast_linear_error_alltok": "ε", "cka_embed": "G-CKA",
         "cosine_sim": "Cos.", "orig_head_accuracy": "Head", "kl_div": "KL"}

print("(a) Within-length ranking, ρ_len")
table = []
for group, combos in GROUPS.items():
    for model, dataset in combos:
        row = {"Model / dataset": label(model, dataset)}
        for metric, short in SHORT.items():
            row[short] = get(model, dataset, metric, "rho_len")
        row["Fewest ρ"] = get(model, dataset, "fewest", "rho")
        table.append(row)
    mean_row = pd.DataFrame(table[-len(combos):]).drop(columns="Model / dataset").mean()
    table.append({"Model / dataset": f"Mean ({group})", **mean_row.to_dict()})
display(pd.DataFrame(table).set_index("Model / dataset").map(f2))

print("(b) Top-5 accuracy drop on natural images (pp)")
table = []
for model, dataset in NATURAL:
    row = {"Model / dataset": label(model, dataset)}
    for metric, short in SHORT.items():
        row[short] = get(model, dataset, metric, "top5")
    row["Fewest"] = get(model, dataset, "fewest", "top5")
    row["Oracle"] = get(model, dataset, "oracle", "top5")
    row["SD"] = get(model, dataset, "seed_sd_top5", "top5")
    table.append(row)
table.append({"Model / dataset": "Mean", **pd.DataFrame(table).drop(columns="Model / dataset").mean().to_dict()})
display(pd.DataFrame(table).set_index("Model / dataset").map(f2))

# Caption check: "cosine similarity on the same rows: rho_len = 0.82, top-5 drop 1.75 pp"
head_combos = R[R.metric == "orig_head_accuracy"][["model", "dataset"]].values.tolist()
cos_rows = [(get(m, d, "cosine_sim", "rho_len"), get(m, d, "cosine_sim", "top5")) for m, d in head_combos]
print(f"cosine similarity on the {len(head_combos)} head rows: rho_len = {np.mean([r[0] for r in cos_rows]):.2f}, "
      f"top-5 = {np.mean([r[1] for r in cos_rows]):.2f} pp")

(a) Within-length ranking, ρ_len


,Ang.,L-CKA,ε,G-CKA,Cos.,Head,KL,Fewest ρ
Model / dataset,,,,,,,,
DeiT-S / C100,0.93,0.71,0.89,0.91,0.61,–,–,0.69
DeiT-S / IN-1k,0.22,0.01,0.24,0.22,0.65,0.74,0.46,0.90
ViT-S / C100,0.58,-0.30,0.25,0.63,0.55,–,–,0.94
ViT-S / IN-1k,0.91,-0.70,0.09,0.96,0.96,0.93,0.98,0.86
DINOv2-B / C100,0.14,0.44,0.16,0.82,0.87,–,–,0.89
DINOv2-B / IN-1k,0.24,0.51,0.13,0.87,0.94,–,–,0.84
ViT-L / C100,0.55,0.58,0.63,0.41,0.39,–,–,0.61
ViT-L / IN-1k,0.42,0.57,0.37,0.83,0.85,0.71,0.83,0.74
Mean (Natural),0.50,0.23,0.35,0.71,0.73,0.80,0.76,0.81


(b) Top-5 accuracy drop on natural images (pp)


,Ang.,L-CKA,ε,G-CKA,Cos.,Head,KL,Fewest,Oracle,SD
Model / dataset,,,,,,,,,,
DeiT-S / C100,0.17,0.53,-0.12,0.17,0.68,–,–,2.16,-0.13,0.12
DeiT-S / IN-1k,2.67,2.64,3.42,3.24,1.83,1.04,2.15,3.11,1.04,0.12
ViT-S / C100,3.12,6.42,4.89,3.34,3.34,–,–,3.75,2.61,0.16
ViT-S / IN-1k,3.26,12.38,8.79,2.94,2.94,2.94,2.94,6.28,2.94,0.13
DINOv2-B / C100,1.90,2.83,4.80,1.35,1.35,–,–,2.66,1.32,0.25
DINOv2-B / IN-1k,1.88,6.85,13.76,1.46,1.46,–,–,5.00,1.46,0.20
ViT-L / C100,0.19,0.03,0.12,0.17,0.17,–,–,3.08,-0.43,0.24
ViT-L / IN-1k,0.77,0.76,0.93,0.61,0.49,0.73,0.63,6.00,0.34,0.18
Mean,1.75,4.05,4.57,1.66,1.53,1.57,1.91,4.00,1.14,0.18


cosine similarity on the 3 head rows: rho_len = 0.82, top-5 = 1.75 pp


## Table C.3 — Best and worst spans per combination (p. 20)

Sort all spans of a combination by their true drop: the first 5 are the "best spans (oracle)", the last one is the worst.
No metric involved, only the accuracy / AUROC files. **Original code:** one-off scripts, not kept in this repo; this notebook is the reference implementation.

In [9]:
table = []
for group, combos in GROUPS.items():
    for model, dataset in combos:
        original, drops = load_drops(model, dataset)
        drops = drops.sort_values("drop", kind="stable")
        cell = lambda r: f"{r.start}->{r.end} ({r.drop:.2f})"
        best5 = [cell(r) for r in drops.head(5).itertuples()]
        worst = cell(next(drops.tail(1).itertuples()))
        table.append({"Group": group, "Model / dataset": label(model, dataset), "Original": f2(original),
                      "1st": best5[0], "2nd": best5[1], "3rd": best5[2], "4th": best5[3], "5th": best5[4], "Worst": worst})
pd.DataFrame(table).set_index(["Group", "Model / dataset"])

Original             1st             2nd            3rd             4th             5th          Worst
Group           Model / dataset                                                                                                                 
Natural         DeiT-S / C100                71.24  10->11 (-0.53)   9->11 (-0.46)  9->10 (-0.31)    8->10 (0.17)     7->8 (0.49)   0->4 (25.50)
                DeiT-S / IN-1k               73.79   10->11 (0.10)     2->3 (0.71)    3->4 (0.95)    9->10 (1.69)     4->5 (1.75)   0->4 (47.18)
                ViT-S / C100                 81.95     4->5 (1.96)     3->4 (2.32)    2->3 (2.42)     5->6 (2.43)     6->7 (3.92)   0->4 (38.01)
                ViT-S / IN-1k                73.57     3->4 (1.95)     2->3 (2.19)    4->5 (2.67)     1->2 (3.33)     5->6 (4.54)   0->4 (58.48)
                DINOv2-B / C100              86.96     0->1 (0.71)     2->3 (0.94)    5->6 (1.59)     1->2 (1.60)     8->9 (1.77)   4->8 (46.47)
                DINOv2-B / IN-1k             74.77     2->3 (1.17)     0->1 (1.25)    4->5 (1.53)     3->4 (1.57)     5->6 (1.77)   4->8 (62.99)
                ViT-L / C100                 85.78  21->22 (-0.63)  21->23 (-0.53)   8->9 (-0.43)  22->23 (-0.29)  20->23 (-0.29)   2->6 (55.62)
                ViT-L / IN-1k                78.56     3->4 (0.08)   22->23 (0.26)  16->17 (0.43)     6->7 (0.43)    9->10 (0.49)   2->6 (72.43)
Domain-specific RAD-DINO / CXR14             81.90    5->9 (-1.70)   7->10 (-1.63)  6->10 (-1.57)    6->9 (-1.53)   8->11 (-1.37)    0->4 (1.00)
                RAD-DINO / Pneum.            99.39    6->8 (-0.13)    3->5 (-0.07)  7->11 (-0.06)    2->4 (-0.06)    5->8 (-0.06)    0->4 (0.29)
                RAD-DINO-MAIRA-2 / CXR14     75.79   8->11 (-1.47)   7->11 (-1.07)  9->11 (-1.03)    5->9 (-0.91)    6->9 (-0.83)    0->4 (1.10)
                RAD-DINO-MAIRA-2 / Pneum.    98.99    7->9 (-0.18)    7->8 (-0.16)   6->8 (-0.07)    6->9 (-0.05)   9->10 (-0.04)    1->5 (1.14)
                Phikon-v2 / PCam             96.15  19->21 (-0.88)  20->21 (-0.60)   0->3 (-0.54)  15->17 (-0.54)    0->2 (-0.50)  17->20 (1.65)
                Phikon-v2 / TCGA-UT          98.68    3->5 (-0.02)    3->4 (-0.02)   4->5 (-0.02)  13->14 (-0.01)    1->3 (-0.01)  17->21 (0.44)
General-purpose DINOv2-B / CXR14             76.97    6->7 (-0.86)    4->6 (-0.86)   5->6 (-0.72)    5->7 (-0.66)    3->4 (-0.65)    4->8 (9.00)
                DINOv2-B / Pneum.            95.31    3->4 (-0.22)    4->6 (-0.09)  9->10 (-0.05)    1->2 (-0.01)    8->10 (0.04)    6->9 (3.23)
                ViT-L / PCam                 93.16  21->23 (-1.01)   6->10 (-1.00)  7->11 (-0.99)   8->11 (-0.97)   7->10 (-0.88)    1->5 (5.67)
                ViT-L / TCGA-UT              94.86   8->12 (-0.61)   7->10 (-0.52)  9->13 (-0.51)   7->11 (-0.51)   9->12 (-0.47)    1->5 (6.38)

---
# Section 4.2: robustness (Table 2)

## Table 2 — Robustness on medical data (p. 5)

For each medical combination:
- **Original**: AUROC of the unapproximated model.
- **Within 1 pp**: share of all candidate spans (38 or 86) whose drop is below 1 pp.
- **Worst drop**: largest drop of any span.
- **d̄_adj**: average angular distance over the single-block spans (0→1, 1→2, …). Small = neighbouring blocks are
  similar = the representation changes gradually.

**Files:** `experiments/4_1-4_2_choosing_blocks/medical/accuracies/combined/`, `experiments/4_1-4_2_choosing_blocks/medical/analysis/output_level_eval_s4_*.csv`.
**Original code:** none saved; computed here from the definitions above.

In [10]:
# Worked example: RAD-DINO / ChestX-ray14
original, drops = load_drops("raddino", "chestxray14multilabel")
print(f"original AUROC = {original:.2f}")
print(f"{(drops['drop'] < 1).sum()} of {len(drops)} spans have a drop below 1 pp -> {100 * (drops['drop'] < 1).mean():.0f}%")
print("spans with drop >= 1 pp:")
display(drops[drops["drop"] >= 1].round(3))
metrics = load_metrics("raddino", "chestxray14multilabel")
single = metrics[metrics.end - metrics.start == 1][["start", "end", "angular_dist"]]
display(single.round(3).T)
print(f"d_adj = mean of these = {single.angular_dist.mean():.2f}")

original AUROC = 81.90
37 of 38 spans have a drop below 1 pp -> 97%
spans with drop >= 1 pp:


,start,end,length,drop,seed_sd
30,0,4,4,1.001,0.204


,0,4,8,12,16,20,24,28,32,35,37
start,0.000,1.000,2.000,3.000,4.000,5.00,6.000,7.000,8.000,9.000,10.00
end,1.000,2.000,3.000,4.000,5.000,6.00,7.000,8.000,9.000,10.000,11.00
angular_dist,0.133,0.103,0.112,0.098,0.097,0.09,0.085,0.082,0.077,0.064,0.09


d_adj = mean of these = 0.09


In [11]:
TABLE2_ORDER = [("dinov2base", "chestxray14multilabel"), ("raddino", "chestxray14multilabel"), ("raddinomaira2", "chestxray14multilabel"),
                ("dinov2base", "chestxraypneumonia"), ("raddino", "chestxraypneumonia"), ("raddinomaira2", "chestxraypneumonia"),
                ("vitlarge", "patchcamelyon"), ("phikonv2", "patchcamelyon"), ("vitlarge", "tcgaut"), ("phikonv2", "tcgaut")]
table = []
for model, dataset in TABLE2_ORDER:
    original, drops = load_drops(model, dataset)
    metrics = load_metrics(model, dataset)
    single_blocks = metrics[metrics.end - metrics.start == 1]
    table.append({"Model": MODEL_NAME[model], "Dataset": DATASET_NAME[dataset],
                  "Original": f2(original),
                  "Within 1 pp": f"{100 * (drops['drop'] < 1).mean():.0f}%",
                  "Worst drop": f2(drops["drop"].max()),
                  "d_adj": f2(single_blocks["angular_dist"].mean()),
                  "(n spans)": len(drops)})
pd.DataFrame(table)

,Model,Dataset,Original,Within 1 pp,Worst drop,d_adj,(n spans)
0,DINOv2-B,CXR14,76.97,61%,9.00,0.27,38
1,RAD-DINO,CXR14,81.90,97%,1.00,0.09,38
2,RAD-DINO-MAIRA-2,CXR14,75.79,97%,1.10,0.13,38
3,DINOv2-B,Pneum.,95.31,45%,3.23,0.27,38
4,RAD-DINO,Pneum.,99.39,100%,0.29,0.09,38
5,RAD-DINO-MAIRA-2,Pneum.,98.99,95%,1.14,0.14,38
6,ViT-L,PCam,93.16,79%,5.67,0.10,86
7,Phikon-v2,PCam,96.15,95%,1.65,0.07,86
8,ViT-L,TCGA-UT,94.86,80%,6.38,0.10,86
9,Phikon-v2,TCGA-UT,98.68,100%,0.44,0.06,86


---
# Section 4.3: two-block approximations (Tables 3, C.4, C.5, C.6)

**The question:** can we predict the drop of replacing two blocks from information about single blocks, without running the pair?

Two kinds of pairs:
- **contiguous**: one span s→s+2 (two neighbouring blocks, one translator). E.g. 3→5 = blocks 3→4 and 4→5 together.
- **disjoint**: two separate single blocks a→a+1 and b→b+1, each with its own translator (ImageNet-1k only).

Three predictors (all: lower value = predicted safer):
- **Angular distance** (one forward pass): contiguous → angular distance of the span s→s+2 itself;
  disjoint → angular distance of block 1 + angular distance of block 2.
- **Cosine similarity** (one run per single block, no labels): (1 − cosine of block 1) + (1 − cosine of block 2).
- **Summed drops** (needs labels): measured drop of block 1 + measured drop of block 2.

How each predictor is judged on one model / dataset:
- **ρ** = Spearman(predictor, true drop) over all pairs.
- **Regret** = true drop of the pair the predictor ranks safest − true drop of the actually best pair. 0 = it found the best.
- **Random regret** = average true drop − best true drop (what you lose on average by picking at random).

Additivity (does the sum predict the size, not just the order?):
- **R²** of the true drop against the summed drops (not fitted, so it can be negative).
- **Δ > Δ̂**: share of pairs whose true drop is larger than the sum.

**Files (natural):** `experiments/4_3_combining/accuracies/combined/*_{contiguous,disjoint}.csv` (true pair drops),
`experiments/4_3_combining/analysis/cka_angular_1_2_layer_spans.csv` (angular distances),
s2 files (original score, single-block drops, cosine).
**Files (medical):** the s4 files; the contiguous pairs are the length-2 spans of Table 1.
**Original code:** one-off scripts, not kept in this repo; this notebook is the reference implementation.

The angular distances for this section are read from `cka_angular_1_2_layer_spans.csv`, as in the thesis.

In [12]:
# Angular distance of every 1- and 2-block span (natural images), as used in s3.
s3_angular_file = pd.read_csv(S3 / "analysis" / "cka_angular_1_2_layer_spans.csv")
S3_ANGULAR = {}
for r in s3_angular_file.itertuples():
    S3_ANGULAR[(r.model, r.dataset, r.start, r.end)] = r.angular_dist


def natural_pairs(model, dataset, shape):
    # One row per two-block approximation on natural images. shape = "contiguous" or "disjoint".
    original, drops = load_drops(model, dataset)
    single_drop = {(r.start, r.end): r.drop for r in drops[drops.length == 1].itertuples()}
    one_minus_cos = {(r.start, r.end): 1 - r.cosine_sim for r in load_metrics(model, dataset).itertuples()}
    ang = lambda s, e: S3_ANGULAR[(model, dataset, s, e)]

    acc = pd.read_csv(S3 / "accuracies" / "combined" / f"accuracies_results_{model}_{dataset}_{shape}.csv")
    rows = []
    for _, r in acc.iterrows():
        spans = parse_spans(r["approx_layer"])
        if shape == "contiguous":
            (s, e), = spans                        # e.g. (3, 5)
            block1, block2 = (s, s + 1), (s + 1, e)  # (3, 4) and (4, 5)
            angular = ang(s, e)                     # angular distance of the 2-block span itself
        else:
            block1, block2 = sorted(spans)          # e.g. (0, 1) and (4, 5)
            angular = ang(*block1) + ang(*block2)
        rows.append({"pair": f"{block1[0]}->{block1[1]} + {block2[0]}->{block2[1]}",
                     "true drop": original - r["accuracy_mean"] * 100,
                     "angular": angular,
                     "cosine": one_minus_cos[block1] + one_minus_cos[block2],
                     "summed drops": single_drop[block1] + single_drop[block2],
                     # the separate parts, used in Table C.6 and the distance check under Table C.5
                     "ang 1": ang(*block1), "ang 2": ang(*block2),
                     "cos 1": one_minus_cos[block1], "cos 2": one_minus_cos[block2],
                     "blocks between": block2[0] - block1[1]})
    return pd.DataFrame(rows)


def medical_pairs(model, dataset):
    # One row per contiguous pair s->s+2 on medical data (the length-2 spans of Table 1).
    original, drops = load_drops(model, dataset)
    single_drop = {(r.start, r.end): r.drop for r in drops[drops.length == 1].itertuples()}
    metrics = load_metrics(model, dataset)
    angular = {(r.start, r.end): r.angular_dist for r in metrics.itertuples()}
    one_minus_cos = {(r.start, r.end): 1 - r.cosine_sim for r in metrics.itertuples()}
    rows = []
    for r in drops[drops.length == 2].itertuples():
        block1, block2 = (r.start, r.start + 1), (r.start + 1, r.end)
        rows.append({"pair": f"{r.start}->{r.end}", "true drop": r.drop,
                     "angular": angular[(r.start, r.end)],
                     "cosine": one_minus_cos[block1] + one_minus_cos[block2],
                     "summed drops": single_drop[block1] + single_drop[block2]})
    return pd.DataFrame(rows)


def judge_predictors(pairs):
    # rho and regret of each predictor, random regret, and the additivity numbers (see the explanation above).
    best = pairs["true drop"].min()
    out = {}
    for predictor in ["angular", "cosine", "summed drops"]:
        out[f"{predictor} ρ"] = spearman(pairs[predictor], pairs["true drop"])
        pick = pairs[predictor].idxmin()  # the pair this predictor thinks is safest
        out[f"{predictor} regret"] = pairs.loc[pick, "true drop"] - best
    out["random regret"] = pairs["true drop"].mean() - best
    out["R²"] = r2_score(pairs["true drop"], pairs["summed drops"])
    out["% drop > sum"] = 100 * (pairs["true drop"] > pairs["summed drops"]).mean()
    out["mean drop − sum"] = (pairs["true drop"] - pairs["summed drops"]).mean()
    return out

### Worked example: DeiT-S / ImageNet-1k, contiguous pairs

All 10 pairs with their true drop and the three predictors, sorted by true drop (so the first row is the best pair).

In [13]:
pairs = natural_pairs("deitsmall", "imagenet1k", "contiguous")
display(pairs[["pair", "true drop", "angular", "cosine", "summed drops"]].sort_values("true drop").round(3))
best = pairs["true drop"].min()
print(f"best pair: {pairs.loc[pairs['true drop'].idxmin(), 'pair']} with drop {best:.2f} pp")
for predictor in ["angular", "cosine", "summed drops"]:
    pick = pairs[predictor].idxmin()
    print(f"{predictor:13s} picks {pairs.loc[pick, 'pair']:16s} drop {pairs.loc[pick, 'true drop']:.2f} -> regret {pairs.loc[pick, 'true drop'] - best:.2f} pp")
print(f"random pick: average drop {pairs['true drop'].mean():.2f} -> regret {pairs['true drop'].mean() - best:.2f} pp")

,pair,true drop,angular,cosine,summed drops
2,2->3 + 3->4,4.090,0.245,0.183,1.660
9,9->10 + 10->11,4.097,0.151,0.192,1.790
3,3->4 + 4->5,5.253,0.242,0.161,2.703
8,8->9 + 9->10,7.400,0.151,0.221,5.513
1,1->2 + 2->3,7.643,0.277,0.240,4.327
4,4->5 + 5->6,9.267,0.240,0.160,4.713
5,5->6 + 6->7,10.850,0.229,0.180,6.357
6,6->7 + 7->8,13.057,0.210,0.209,7.747
7,7->8 + 8->9,13.213,0.195,0.225,8.180
0,0->1 + 1->2,32.130,0.325,0.388,14.480


best pair: 2->3 + 3->4 with drop 4.09 pp
angular       picks 8->9 + 9->10     drop 7.40 -> regret 3.31 pp
cosine        picks 4->5 + 5->6      drop 9.27 -> regret 5.18 pp
summed drops  picks 2->3 + 3->4      drop 4.09 -> regret 0.00 pp
random pick: average drop 10.70 -> regret 6.61 pp


### Compute all pairs for every combination

In [14]:
PAIRS = {}  # (setting, model, dataset) -> table of pairs
for model, dataset in NATURAL:
    PAIRS[("Natural, contiguous", model, dataset)] = natural_pairs(model, dataset, "contiguous")
    if dataset == "imagenet1k":
        PAIRS[("Natural, disjoint", model, dataset)] = natural_pairs(model, dataset, "disjoint")
for model, dataset in DOMAIN_SPECIFIC:
    PAIRS[("Domain-specific, contiguous", model, dataset)] = medical_pairs(model, dataset)
for model, dataset in GENERAL_PURPOSE:
    PAIRS[("General-purpose, contiguous", model, dataset)] = medical_pairs(model, dataset)

rows = []
for (setting, model, dataset), pairs in PAIRS.items():
    rows.append({"setting": setting, "combo": label(model, dataset), "n pairs": len(pairs), **judge_predictors(pairs)})
P = pd.DataFrame(rows)
SETTINGS = ["Natural, contiguous", "Natural, disjoint", "Domain-specific, contiguous", "General-purpose, contiguous"]

print("per combination (Table 3 and C.4 are averages of this):")
display(P.round(2))

# Sanity check: the s3 contiguous pairs should be the same runs as the s2 length-2 spans ("the length-2 spans of Section 4.1")
for model, dataset in NATURAL:
    pairs = PAIRS[("Natural, contiguous", model, dataset)]
    s2_drops = load_drops(model, dataset)[1]
    s2_len2 = {f"{r.start}->{r.start + 1} + {r.start + 1}->{r.end}": r.drop for r in s2_drops[s2_drops.length == 2].itertuples()}
    biggest_difference = max(abs(r["true drop"] - s2_len2[r["pair"]]) for _, r in pairs.iterrows())
    print(f"{label(model, dataset):18s} largest |s3 pair drop - s2 length-2 drop| = {biggest_difference:.4f} pp")

per combination (Table 3 and C.4 are averages of this):


,setting,combo,n pairs,angular ρ,angular regret,cosine ρ,cosine regret,summed drops ρ,summed drops regret,random regret,R²,% drop > sum,mean drop − sum
0,"Natural, contiguous",DeiT-S / C100,10,0.89,0.00,0.84,0.64,1.00,0.00,6.22,0.84,90.00,1.88
1,"Natural, contiguous",DeiT-S / IN-1k,10,0.12,3.31,0.45,5.18,0.96,0.00,6.61,0.28,100.00,4.95
2,"Natural, disjoint",DeiT-S / IN-1k,45,0.60,6.24,0.76,0.96,0.95,0.00,6.35,0.89,84.44,0.98
3,"Natural, contiguous",ViT-S / C100,10,0.72,0.00,0.67,2.10,0.99,0.00,4.69,-0.03,100.00,2.81
4,"Natural, contiguous",ViT-S / IN-1k,10,0.94,0.28,0.87,0.00,0.93,0.00,10.38,0.51,100.00,4.46
5,"Natural, disjoint",ViT-S / IN-1k,45,0.88,2.07,0.79,0.00,0.93,0.00,7.88,0.74,80.00,1.46
6,"Natural, contiguous",DINOv2-B / C100,10,0.52,0.00,0.82,0.61,0.92,0.61,5.03,0.23,90.00,3.31
7,"Natural, contiguous",DINOv2-B / IN-1k,10,0.38,0.00,0.71,1.76,0.76,1.76,10.15,0.53,90.00,4.88
8,"Natural, disjoint",DINOv2-B / IN-1k,45,0.69,1.01,0.94,0.46,0.98,0.46,7.95,0.96,75.56,0.67
9,"Natural, contiguous",ViT-L / C100,22,0.39,0.86,0.27,1.44,0.95,0.00,6.17,0.37,95.45,-0.73


DeiT-S / C100      largest |s3 pair drop - s2 length-2 drop| = 0.0000 pp
DeiT-S / IN-1k     largest |s3 pair drop - s2 length-2 drop| = 0.0000 pp
ViT-S / C100       largest |s3 pair drop - s2 length-2 drop| = 0.0000 pp
ViT-S / IN-1k      largest |s3 pair drop - s2 length-2 drop| = 0.0000 pp
DINOv2-B / C100    largest |s3 pair drop - s2 length-2 drop| = 0.0000 pp
DINOv2-B / IN-1k   largest |s3 pair drop - s2 length-2 drop| = 0.0000 pp
ViT-L / C100       largest |s3 pair drop - s2 length-2 drop| = 0.0000 pp


ViT-L / IN-1k      largest |s3 pair drop - s2 length-2 drop| = 0.0000 pp


## Table 3 — Planning two-block approximations (p. 7)

Mean over the combinations of each setting (8 natural contiguous, 4 natural disjoint, 6 domain-specific, 4 general-purpose).

In [15]:
table = []
for setting in SETTINGS:
    sel = P[P.setting == setting]
    table.append({"": setting,
                  "Ang ρ": f2(sel["angular ρ"].mean()), "Ang regret": f2(sel["angular regret"].mean()),
                  "Cos ρ": f2(sel["cosine ρ"].mean()), "Cos regret": f2(sel["cosine regret"].mean()),
                  "Sum ρ": f2(sel["summed drops ρ"].mean()), "Sum regret": f2(sel["summed drops regret"].mean()),
                  "Random regret": f2(sel["random regret"].mean()),
                  "R²": f2(sel["R²"].mean()), "Δ > Δ̂": f"{sel['% drop > sum'].mean():.0f}%"})
pd.DataFrame(table).set_index("")

,Ang ρ,Ang regret,Cos ρ,Cos regret,Sum ρ,Sum regret,Random regret,R²,Δ > Δ̂
,,,,,,,,,
"Natural, contiguous",0.55,0.64,0.69,1.47,0.91,0.38,7.23,0.38,95%
"Natural, disjoint",0.56,2.86,0.79,0.57,0.95,0.12,8.45,0.89,78%
"Domain-specific, contiguous",0.30,0.26,-0.19,0.66,0.87,0.13,0.53,0.70,60%
"General-purpose, contiguous",0.24,0.98,0.49,0.42,0.72,0.21,1.17,0.20,68%


## Table C.4 — Planning two-block approximations, mean (SD) (p. 20)

Same numbers as Table 3, with the SD across combinations in brackets.

In [16]:
mean_sd = lambda x: f"{x.mean():.2f} ({x.std():.2f})"
table = []
for setting in SETTINGS:
    sel = P[P.setting == setting]
    for predictor, nice_name in [("angular", "Angular distance"), ("cosine", "Cosine similarity"), ("summed drops", "Summed drops")]:
        table.append({"Setting": setting, "Predictor": nice_name,
                      "ρ": mean_sd(sel[f"{predictor} ρ"]), "Regret (pp)": mean_sd(sel[f"{predictor} regret"])})
    table.append({"Setting": setting, "Predictor": "Random", "ρ": "–", "Regret (pp)": mean_sd(sel["random regret"])})
pd.DataFrame(table).set_index(["Setting", "Predictor"])

ρ  Regret (pp)
Setting                     Predictor                                   
Natural, contiguous         Angular distance    0.55 (0.28)  0.64 (1.13)
                            Cosine similarity   0.69 (0.22)  1.47 (1.68)
                            Summed drops        0.91 (0.09)  0.38 (0.63)
                            Random                        –  7.23 (2.21)
Natural, disjoint           Angular distance    0.56 (0.34)  2.86 (2.31)
                            Cosine similarity   0.79 (0.11)  0.57 (0.44)
                            Summed drops        0.95 (0.03)  0.12 (0.23)
                            Random                        –  8.45 (2.23)
Domain-specific, contiguous Angular distance    0.30 (0.50)  0.26 (0.18)
                            Cosine similarity  -0.19 (0.68)  0.66 (0.52)
                            Summed drops        0.87 (0.08)  0.13 (0.18)
                            Random                        –  0.53 (0.40)
General-purpose, contiguous Angular distance    0.24 (0.16)  0.98 (0.56)
                            Cosine similarity   0.49 (0.15)  0.42 (0.26)
                            Summed drops        0.72 (0.32)  0.21 (0.27)
                            Random                        –  1.17 (0.32)

## Table C.5 — How additive are two-block drops? (p. 21)

R² of the unfitted sum (mean (SD) across combinations), share of pairs whose drop exceeds the sum, and mean (drop − sum).

Below it, the text claim in Sec. 4.3: for disjoint pairs, R² = 0.80 with 1–2 blocks between them, 0.86 with 3–4, 0.97
with 5 or more. "Blocks between" = untouched blocks between the two replaced ones (0→1 + 2→3 has 1 between).
R² is computed per model and bin, then averaged over the 4 models.

In [17]:
table = []
for setting in SETTINGS:
    sel = P[P.setting == setting]
    table.append({"": setting, "R²": f"{sel['R²'].mean():.2f} ({sel['R²'].std():.2f})",
                  "Drop > sum": f"{sel['% drop > sum'].mean():.0f}%", "Mean drop − sum (pp)": f2(sel["mean drop − sum"].mean())})
display(pd.DataFrame(table).set_index(""))

rows = []
for (setting, model, dataset), pairs in PAIRS.items():
    if setting != "Natural, disjoint":
        continue
    for bin_name, low, high in [("1-2 between", 1, 2), ("3-4 between", 3, 4), ("5+ between", 5, 99)]:
        sel = pairs[(pairs["blocks between"] >= low) & (pairs["blocks between"] <= high)]
        rows.append({"model": MODEL_NAME[model], "bin": bin_name, "n pairs": len(sel),
                     "R²": r2_score(sel["true drop"], sel["summed drops"])})
by_distance = pd.DataFrame(rows)
display(by_distance.pivot(index="model", columns="bin", values="R²").round(2))
print("mean over models:", by_distance.groupby("bin")["R²"].mean().round(2).to_dict())

,R²,Drop > sum,Mean drop − sum (pp)
,,,
"Natural, contiguous",0.38 (0.26),95%,2.39
"Natural, disjoint",0.89 (0.11),78%,0.78
"Domain-specific, contiguous",0.70 (0.31),60%,0.01
"General-purpose, contiguous",0.20 (0.18),68%,0.20


bin,1-2 between,3-4 between,5+ between
model,,,
DINOv2-B,0.94,0.98,0.99
DeiT-S,0.57,0.91,0.99
ViT-L,1.00,1.00,1.00
ViT-S,0.71,0.54,0.90


mean over models: {'1-2 between': 0.8, '3-4 between': 0.86, '5+ between': 0.97}


## Table C.6 — Adding similarity terms to the summed drops (p. 22)

Can we beat the plain sum by fitting a small correction? For each combination, fit
`true drop ≈ w1·(term of block 1) + w2·(term of block 2) + w3·(summed drops) + b`, with term = angular distance or (1 − cosine).
To keep it honest, each pair is predicted by a regression fitted on all the *other* pairs (leave-one-out).
Then ρ and R² of those predictions, as mean (SD) and median across combinations. The plain sum is not fitted.
Natural images only. **Original code:** one-off scripts, not kept in this repo; this notebook is the reference implementation.

In [18]:
rows = []
for (setting, model, dataset), pairs in PAIRS.items():
    if not setting.startswith("Natural"):
        continue
    shape = setting.split(", ")[1]
    y = pairs["true drop"]
    rows.append({"shape": shape, "predictor": "Summed drops (unfitted)",
                 "ρ": spearman(pairs["summed drops"], y), "R²": r2_score(y, pairs["summed drops"])})
    for nice_name, columns in [("+ angular distance (fitted)", ["ang 1", "ang 2", "summed drops"]),
                               ("+ cosine similarity (fitted)", ["cos 1", "cos 2", "summed drops"])]:
        predicted = cross_val_predict(LinearRegression(), pairs[columns], y, cv=LeaveOneOut())
        rows.append({"shape": shape, "predictor": nice_name, "ρ": spearman(predicted, y), "R²": r2_score(y, predicted)})
C6 = pd.DataFrame(rows)

table = []
for (shape, predictor), sel in C6.groupby(["shape", "predictor"], sort=False):
    table.append({"Shape": shape, "Predictor": predictor,
                  "ρ": f"{sel['ρ'].mean():.2f} ({sel['ρ'].std():.2f})",
                  "R²": f"{sel['R²'].mean():.2f} ({sel['R²'].std():.2f})", "R² median": f2(sel["R²"].median())})
pd.DataFrame(table).set_index(["Shape", "Predictor"])

ρ            R² R² median
Shape      Predictor                                                        
contiguous Summed drops (unfitted)       0.91 (0.09)   0.38 (0.26)      0.32
           + angular distance (fitted)   0.66 (0.28)   0.22 (0.59)      0.26
           + cosine similarity (fitted)  0.62 (0.39)  -0.11 (1.13)      0.30
disjoint   Summed drops (unfitted)       0.95 (0.03)   0.89 (0.11)      0.92
           + angular distance (fitted)   0.96 (0.02)   0.96 (0.04)      0.97
           + cosine similarity (fitted)  0.96 (0.03)   0.97 (0.02)      0.96

---
# Section 4.4: translator fitted on another dataset (Tables 4, C.7, C.8)

Setting: single-block approximations on the 10 medical combinations. The translator is fitted on a **source** dataset:
- **Target**: the dataset it is scored on (reference),
- **In-domain**: the other dataset of the same domain (ChestX-ray14 ↔ Pneumonia, PCam ↔ TCGA-UT),
- **CIFAR-100**.

Two head settings: **retrained** (new linear head trained on the target on top of the approximated model, column
`auroc`) and **frozen** (head trained once on the original model and reused, column `frozen_head_auroc`).

## Table 4 — Fitting the translator on a source dataset (p. 8)

Per model, head setting and source:
1. For every seed, every single block, and both datasets of the model's domain: drop = `original_auroc` − score (pp).
2. Per seed: average those drops (over blocks and both datasets) → 3 numbers, one per seed.
3. Cell = mean of the 3 numbers (SD of the 3 numbers).

**Files:** `experiments/4_4_without_target_data/results/*.csv` (one row per seed).
**Original code:** no saved script; same loading as `experiments/4_1-4_2_choosing_blocks/metric_table.ipynb` cell 16.
Some (model, dataset, source, block, seed) rows appear in more than one file; the check below shows whether they ever disagree.

In [19]:
PARTNER = {"chestxray14_multilabel": "chestxray_pneumonia", "chestxray_pneumonia": "chestxray14_multilabel",
           "patchcamelyon": "tcga_ut", "tcga_ut": "patchcamelyon"}
HF_NAME = {"microsoft/rad-dino": "RAD-DINO", "microsoft/rad-dino-maira-2": "MAIRA-2", "owkin/phikon-v2": "Phikon-v2",
           "facebook/dinov2-base": "DINOv2-B", "google/vit-large-patch16-224": "ViT-L"}


def single_block_start(text):
    # '[(4, 5)]' -> 4 ;  '[]' -> -1 (original model) ;  anything else (longer / several spans) -> None
    spans = parse_spans(str(text))
    if len(spans) == 0:
        return -1
    if len(spans) == 1 and spans[0][1] - spans[0][0] == 1:
        return spans[0][0]
    return None


per_seed = pd.concat([pd.read_csv(p) for p in sorted((S5 / "results").glob("*.csv"))], ignore_index=True)
per_seed["block"] = per_seed["approx_layer"].map(single_block_start)
per_seed = per_seed.dropna(subset=["block"])

key = ["model", "dataset", "fit_dataset", "block", "seed"]
duplicates = per_seed[per_seed.duplicated(key, keep=False)]
n_disagree = (duplicates.groupby(key)[["auroc", "frozen_head_auroc"]].nunique() > 1).any(axis=1).sum()
print(f"rows that appear in more than one file: {len(duplicates)}, keys where the copies disagree: {n_disagree}")
per_seed = per_seed.drop_duplicates(key, keep="last")

per_seed = per_seed[(per_seed["block"] >= 0) & (per_seed["dataset"] != "cifar100")]  # single blocks, medical targets


def source_of(row):
    if row["fit_dataset"] == row["dataset"]:
        return "Target"
    if row["fit_dataset"] == PARTNER[row["dataset"]]:
        return "In-domain"
    if row["fit_dataset"] == "cifar100":
        return "CIFAR-100"


per_seed["source"] = per_seed.apply(source_of, axis=1)
per_seed["drop retrained"] = (per_seed["original_auroc"] - per_seed["auroc"]) * 100
per_seed["drop frozen"] = (per_seed["original_auroc"] - per_seed["frozen_head_auroc"]) * 100

rows that appear in more than one file: 138, keys where the copies disagree: 0


In [20]:
# Worked example: DINOv2-B, frozen head, in-domain source
sel = per_seed[(per_seed.model == "facebook/dinov2-base") & (per_seed.source == "In-domain")]
print(f"{len(sel)} rows = 2 datasets x 11 single blocks x 3 seeds")
seed_means = sel.groupby("seed")["drop frozen"].mean()
print("mean drop per seed:", seed_means.round(3).to_dict())
print(f"-> cell = {seed_means.mean():.2f} ({seed_means.std():.2f})   (PDF: 1.93 (0.15))")

66 rows = 2 datasets x 11 single blocks x 3 seeds
mean drop per seed: {0: 1.84, 1: 1.84, 2: 2.106}
-> cell = 1.93 (0.15)   (PDF: 1.93 (0.15))


In [21]:
table = []
for head in ["retrained", "frozen"]:
    for source in ["Target", "In-domain", "CIFAR-100"]:
        row = {"Head": head, "Source": source}
        for hf_key, short in HF_NAME.items():
            sel = per_seed[(per_seed.model == hf_key) & (per_seed.source == source)]
            seed_means = sel.groupby("seed")[f"drop {head}"].mean()
            row[short] = f"{seed_means.mean():.2f} ({seed_means.std():.2f})"
        table.append(row)
display(pd.DataFrame(table).set_index(["Head", "Source"]))

# Text checks (Sec. 4.4)
for head in ["retrained", "frozen"]:
    per_combo = per_seed.groupby(["model", "dataset", "source"])[f"drop {head}"].mean().unstack("source")
    in_domain_cost = per_combo["In-domain"] - per_combo["Target"]
    cifar_cost = (per_combo["CIFAR-100"] - per_combo["Target"]).groupby(level="model").mean()
    print(f"{head}: in-domain minus target, mean over the 10 combinations = {in_domain_cost.mean():.2f} pp")
    print(f"   in-domain minus target per model: {in_domain_cost.groupby(level='model').mean().rename(HF_NAME).round(2).to_dict()}")
    print(f"   CIFAR-100 minus target per model: {cifar_cost.rename(HF_NAME).round(2).to_dict()}")

RAD-DINO       MAIRA-2     Phikon-v2     DINOv2-B        ViT-L
Head      Source                                                                       
retrained Target     -0.10 (0.01)   0.01 (0.02)  -0.01 (0.01)  0.18 (0.09)  0.24 (0.02)
          In-domain  -0.12 (0.01)  -0.06 (0.06)  -0.00 (0.04)  0.33 (0.09)  0.36 (0.03)
          CIFAR-100  -0.16 (0.03)   0.06 (0.05)   0.10 (0.06)  1.41 (0.06)  0.83 (0.03)
frozen    Target     -0.04 (0.01)   0.28 (0.12)   0.01 (0.01)  1.30 (0.08)  2.50 (0.08)
          In-domain  -0.02 (0.00)   0.12 (0.05)   0.19 (0.05)  1.93 (0.15)  3.09 (0.16)
          CIFAR-100   0.23 (0.02)   0.69 (0.07)   0.64 (0.04)  9.78 (0.33)  6.13 (0.24)

retrained: in-domain minus target, mean over the 10 combinations = 0.04 pp
   in-domain minus target per model: {'DINOv2-B': 0.15, 'ViT-L': 0.12, 'RAD-DINO': -0.02, 'MAIRA-2': -0.06, 'Phikon-v2': 0.01}
   CIFAR-100 minus target per model: {'DINOv2-B': 1.23, 'ViT-L': 0.59, 'RAD-DINO': -0.06, 'MAIRA-2': 0.06, 'Phikon-v2': 0.11}
frozen: in-domain minus target, mean over the 10 combinations = 0.25 pp
   in-domain minus target per model: {'DINOv2-B': 0.63, 'ViT-L': 0.59, 'RAD-DINO': 0.02, 'MAIRA-2': -0.16, 'Phikon-v2': 0.18}
   CIFAR-100 minus target per model: {'DINOv2-B': 8.48, 'ViT-L': 3.63, 'RAD-DINO': 0.26, 'MAIRA-2': 0.41, 'Phikon-v2': 0.63}


## Table C.7 — Do drop rankings carry over between datasets? (p. 22)

For one model, take the drops of the same spans on its two datasets and compute Spearman between them
(both use translators fitted on their own dataset, as in Table 1).
**Within length**: Spearman per span length, averaged over lengths 1–4. **Single blocks**: Spearman over single blocks only.
**Original code:** one-off scripts, not kept in this repo; this notebook is the reference implementation.

In [22]:
TABLE_C7 = [("deitsmall", "cifar100", "imagenet1k"), ("vitsmall", "cifar100", "imagenet1k"),
            ("dinov2base", "cifar100", "imagenet1k"), ("vitlarge", "cifar100", "imagenet1k"),
            ("dinov2base", "chestxray14multilabel", "chestxraypneumonia"), ("vitlarge", "patchcamelyon", "tcgaut"),
            ("raddino", "chestxray14multilabel", "chestxraypneumonia"),
            ("raddinomaira2", "chestxray14multilabel", "chestxraypneumonia"), ("phikonv2", "patchcamelyon", "tcgaut")]
table = []
for model, dataset1, dataset2 in TABLE_C7:
    drops1 = load_drops(model, dataset1)[1]
    drops2 = load_drops(model, dataset2)[1]
    both = drops1.merge(drops2, on=["start", "end", "length"], suffixes=(" 1", " 2"))  # same spans side by side
    rho_per_length = [spearman(both[both.length == L]["drop 1"], both[both.length == L]["drop 2"]) for L in [1, 2, 3, 4]]
    singles = both[both.length == 1]
    table.append({"Model": MODEL_NAME[model], "Datasets": f"{DATASET_NAME[dataset1]} vs {DATASET_NAME[dataset2]}",
                  "Within length": f2(np.mean(rho_per_length)),
                  "Single blocks": f2(spearman(singles["drop 1"], singles["drop 2"]))})
pd.DataFrame(table)

,Model,Datasets,Within length,Single blocks
0,DeiT-S,C100 vs IN-1k,0.36,0.37
1,ViT-S,C100 vs IN-1k,0.78,0.77
2,DINOv2-B,C100 vs IN-1k,0.89,0.75
3,ViT-L,C100 vs IN-1k,0.48,0.30
4,DINOv2-B,CXR14 vs Pneum.,0.34,0.30
5,ViT-L,PCam vs TCGA-UT,0.64,0.53
6,RAD-DINO,CXR14 vs Pneum.,0.08,0.00
7,RAD-DINO-MAIRA-2,CXR14 vs Pneum.,0.35,0.21
8,Phikon-v2,PCam vs TCGA-UT,0.10,-0.09


## Table C.8 — Choosing a single block without target labels (p. 22)

For each source → target pair (10 pairs) and head setting, rank the single blocks in one of three ways:
- **Random**: regret = average target drop − best target drop.
- **Source labels**: the source's own drops (translator fitted on the source, head trained on the source).
- **Angular distance**: angular distance on the source images (no labels needed).

Every pick is then scored on the **target**, with the translator fitted on the **source** and the target's head.
**Regret** = target drop of the picked block − smallest target drop. **ρ** = Spearman(ranking, target drops).
Rows = mean over the pairs of that model type; "All" = mean over the 10 pairs.

**Files:** `experiments/4_4_without_target_data/accuracies/*.csv` (seed means), s4 `output_level_eval_s4_*.csv` (angular distance).
**Original code:** `experiments/4_4_without_target_data/analysis/s5_random_angdist.py` (its `AngSrc` columns). It rounds every per-pair value to 2 decimals
before averaging, so a mean here can differ by 0.01 (General-purpose retrained angular ρ / regret: 0.28 / 0.40 here, 0.29 / 0.41 in the PDF).

In [23]:
seed_means = pd.concat([pd.read_csv(p) for p in sorted((S5 / "accuracies").glob("*.csv"))], ignore_index=True)
seed_means["block"] = seed_means["approx_layer"].map(single_block_start)
seed_means = seed_means.dropna(subset=["block"]).drop_duplicates(["model", "dataset", "fit_dataset", "block"], keep="last")
SHORT_KEY = {"microsoft/rad-dino": "raddino", "microsoft/rad-dino-maira-2": "raddinomaira2", "owkin/phikon-v2": "phikonv2",
             "facebook/dinov2-base": "dinov2base", "google/vit-large-patch16-224": "vitlarge"}
SCORE_COLUMN = {"retrained": "auroc_mean", "frozen": "frozen_head_auroc_mean"}


def transfer_drops(model, scored_on, fitted_on, head):
    # Drop (pp) of every single block, scored on `scored_on`, with the translator fitted on `fitted_on`.
    rows = seed_means[(seed_means.model == model) & (seed_means.dataset == scored_on)]
    original = rows[rows.block == -1]["auroc_mean"].iloc[0]
    blocks = rows[(rows.fit_dataset == fitted_on) & (rows.block >= 0)].set_index("block")
    return (original - blocks[SCORE_COLUMN[head]]) * 100


def single_block_angular(model, dataset):
    metrics = load_metrics(SHORT_KEY[model], dataset.replace("_", ""))
    return metrics[metrics.end - metrics.start == 1].set_index("start")["angular_dist"]


rows, same_pick = [], []
for model in SHORT_KEY:
    for target, source in PARTNER.items():
        if not ((seed_means.model == model) & (seed_means.dataset == target)).any():
            continue  # this model is not run on this domain
        angular_on_source = single_block_angular(model, source)
        angular_on_target = single_block_angular(model, target)
        same_pick.append(angular_on_source.idxmin() == angular_on_target.idxmin())
        for head in ["retrained", "frozen"]:
            source_drops = transfer_drops(model, source, source, head)  # what you can measure on the source
            target_drops = transfer_drops(model, target, source, head)  # what actually happens on the target
            best = target_drops.min()
            label_pick = source_drops.idxmin()
            angular_pick = angular_on_source.idxmin()
            rows.append({"Model type": "Domain-specific" if SHORT_KEY[model] in ("raddino", "raddinomaira2", "phikonv2") else "General-purpose",
                         "Head": head, "pair": f"{HF_NAME[model]}: {source} -> {target}",
                         "Random regret": target_drops.mean() - best,
                         "Labels ρ": spearman(source_drops, target_drops), "Labels pick": label_pick,
                         "Labels regret": target_drops[label_pick] - best,
                         "Angular ρ": spearman(angular_on_source, target_drops), "Angular pick": angular_pick,
                         "Angular regret": target_drops[angular_pick] - best})
C8 = pd.DataFrame(rows)
print("per pair:")
display(C8.round(2))

per pair:


,Model type,Head,pair,Random regret,Labels ρ,Labels pick,Labels regret,Angular ρ,Angular pick,Angular regret
0,Domain-specific,retrained,RAD-DINO: chestxray_pneumonia -> chestxray14_m...,0.59,0.15,4,0.78,0.98,9,0.23
1,Domain-specific,frozen,RAD-DINO: chestxray_pneumonia -> chestxray14_m...,0.48,0.50,7,0.26,0.57,9,0.34
2,Domain-specific,retrained,RAD-DINO: chestxray14_multilabel -> chestxray_...,0.05,0.43,8,0.02,0.18,9,0.01
3,Domain-specific,frozen,RAD-DINO: chestxray14_multilabel -> chestxray_...,0.05,0.04,8,0.05,-0.23,9,0.12
4,Domain-specific,retrained,MAIRA-2: chestxray_pneumonia -> chestxray14_mu...,0.46,0.45,7,0.42,0.63,9,0.08
5,Domain-specific,frozen,MAIRA-2: chestxray_pneumonia -> chestxray14_mu...,0.32,0.21,1,0.16,-0.01,9,0.15
6,Domain-specific,retrained,MAIRA-2: chestxray14_multilabel -> chestxray_p...,0.22,0.49,8,0.18,0.66,9,0.00
7,Domain-specific,frozen,MAIRA-2: chestxray14_multilabel -> chestxray_p...,0.17,-0.29,2,0.29,0.76,9,0.19
8,Domain-specific,retrained,Phikon-v2: tcga_ut -> patchcamelyon,0.65,-0.12,3,0.69,0.18,22,0.55
9,Domain-specific,frozen,Phikon-v2: tcga_ut -> patchcamelyon,0.81,0.29,21,0.58,0.37,22,0.00


In [24]:
numbers = ["Random regret", "Labels ρ", "Labels regret", "Angular ρ", "Angular regret"]
table = []
for model_type in ["Domain-specific", "General-purpose", "All"]:
    for head in ["retrained", "frozen"]:
        sel = C8[C8.Head == head] if model_type == "All" else C8[(C8.Head == head) & (C8["Model type"] == model_type)]
        table.append({"Model type": model_type, "Head": head, **{n: f2(sel[n].mean()) for n in numbers}})
display(pd.DataFrame(table).set_index(["Model type", "Head"]))
print(f"caption check: angular distance on target images picks the same block as on source images in {sum(same_pick)} of {len(same_pick)} pairs")

Random regret Labels ρ Labels regret Angular ρ Angular regret
Model type      Head                                                                   
Domain-specific retrained          0.34     0.24          0.37      0.45           0.17
                frozen             0.32     0.12          0.24      0.35           0.15
General-purpose retrained          0.72     0.36          0.19      0.28           0.40
                frozen             2.51     0.48          0.41      0.62           0.65
All             retrained          0.49     0.29          0.30      0.38           0.26
                frozen             1.20     0.26          0.31      0.45           0.35

caption check: angular distance on target images picks the same block as on source images in 8 of 10 pairs


---
# Appendix A: sublayers and parameters (Tables A.1, A.2, A.3)

## Table A.1 — Parameter accounting (p. 13)

Standard ViT block, hidden size d (MLP is 4× wider):
- attention: Q, K, V, output projection = 4 · (d·d weights + d biases) = 4d² + 4d
- MLP: d → 4d → d = (4d² + 4d) + (4d² + d) = 8d² + 5d
- two LayerNorms: 4d; DINOv2-family models (DINOv2-B, RAD-DINO, Phikon-v2) also have LayerScale: 2d
- per attention head = attention / number of heads; one linear translator = d · d

The totals cannot come from these formulas alone (embeddings, pooler). As a cross-check, the second table back-solves
the total the code used from `params_saved / params_saved_pct` in the s2 results. Those totals are slightly smaller
than the PDF's (most likely the code divides by the encoder only), and the PDF never quotes the code's percentages.

In [25]:
MODELS_A1 = {"DeiT-S, ViT-S": (384, 6, False), "DINOv2-B, RAD-DINO": (768, 12, True),
             "ViT-L": (1024, 16, False), "Phikon-v2": (1024, 16, True)}  # (d, heads, has LayerScale)
table = []
for name, (d, heads, layer_scale) in MODELS_A1.items():
    attention = 4 * d * d + 4 * d
    mlp = 8 * d * d + 5 * d
    block = attention + mlp + 4 * d + (2 * d if layer_scale else 0)
    table.append({"Models": name, "d": d, "Heads": heads, "Block": block / 1e6, "Attn.": attention / 1e6,
                  "MLP": mlp / 1e6, "Head": attention / heads / 1e6, "Translator": d * d / 1e6})
A1 = pd.DataFrame(table).set_index("Models")
display(A1.round(2))

rows = []
for model, dataset in NATURAL:
    results = pd.read_csv(S2 / "results" / f"results_{model}_{dataset}.csv")
    one_block = results[results["approx_layer"] == "[(1, 2)]"].iloc[0]
    rows.append({"combo": label(model, dataset), "saved by one block (M)": one_block["params_saved"] / 1e6,
                 "total implied by the code (M)": one_block["params_saved"] / one_block["params_saved_pct"] * 100 / 1e6})
display(pd.DataFrame(rows).round(2))

# Text checks: Sec. 4.2 "four blocks of Phikon-v2 ~ 16% of its parameters"; Sec. A.3 "20->23 removes 36.7M, three MLPs 22.0M"
phikon, vitl = A1.loc["Phikon-v2"], A1.loc["ViT-L"]
print(f"Phikon-v2, 4 blocks minus 1 translator = {4 * phikon.Block - phikon.Translator:.2f}M "
      f"= {(4 * phikon.Block - phikon.Translator) / 303.35 * 100:.1f}% of 303.35M")
print(f"ViT-L 20->23 (3 blocks, 1 translator) saves {3 * vitl.Block - vitl.Translator:.1f}M; "
      f"3 MLPs (3 translators) save {3 * (vitl.MLP - vitl.Translator):.1f}M")

,d,Heads,Block,Attn.,MLP,Head,Translator
Models,,,,,,,
"DeiT-S, ViT-S",384,6,1.77,0.59,1.18,0.10,0.15
"DINOv2-B, RAD-DINO",768,12,7.09,2.36,4.72,0.20,0.59
ViT-L,1024,16,12.60,4.20,8.39,0.26,1.05
Phikon-v2,1024,16,12.60,4.20,8.39,0.26,1.05


,combo,saved by one block (M),total implied by the code (M)
0,DeiT-S / C100,1.63,21.29
1,DeiT-S / IN-1k,1.63,21.29
2,ViT-S / C100,1.63,21.29
3,ViT-S / IN-1k,1.63,21.29
4,DINOv2-B / C100,6.50,85.07
5,DINOv2-B / IN-1k,6.50,85.07
6,ViT-L / C100,11.55,302.31
7,ViT-L / IN-1k,11.55,302.31


Phikon-v2, 4 blocks minus 1 translator = 49.34M = 16.3% of 303.35M
ViT-L 20->23 (3 blocks, 1 translator) saves 36.7M; 3 MLPs (3 translators) save 22.0M


## Shared loader for Tables A.2 and A.3 (sublayers)

These come from older per-seed result files (11 Oct, before the Sec. 4 runs; e.g. DINOv2-B / IN-1k original is 74.18
here vs 74.77 in Table C.3). Each row of the output = one configuration, mean (SD) × 100 over its seeds, n = number of seeds.

Which part of the block was replaced, read from three columns:
- `approx_layer` set → **Block** (Identity/Linear from the `translator` column)
- `mlp_linearize` set → **MLP** (Identity/Linear from `mlp_mode`)
- `attn_linearize` set → **Attention** (always identity)
- none set → **Original**

Head-pruning runs (non-empty `head_dict`) are left out. Configurations that were run twice keep the last run per seed;
the printed line says how many duplicates there were and whether they disagreed.

**Files:** `experiments/A2_sublayers/results_full_compare.csv` (DeiT-S, ImageNet), `experiments/A2_sublayers/results_full_amended.csv`
(DINOv2-B and ViT-L, ImageNet), `experiments/A2_sublayers/results_full_table3.csv` (CIFAR-100).
**Original code:** not found; these files were matched to the PDF by value.

In [26]:
def what_was_replaced(row):
    blocks = parse_spans(row["approx_layer"])
    mlps = ast.literal_eval(row["mlp_linearize"])
    attentions = ast.literal_eval(row["attn_linearize"])
    if blocks and not mlps and not attentions:
        return "Block", ", ".join(f"{s}->{e}" for s, e in blocks), row["translator"]
    if mlps and not blocks and not attentions:
        return "MLP", ", ".join(map(str, mlps)), row["mlp_mode"]
    if attentions and not blocks and not mlps:
        return "Attention", ", ".join(map(str, attentions)), row["attn_mode"]
    if not blocks and not mlps and not attentions:
        return "Original", "", "identity"
    return "mixed (not in PDF)", str((blocks, mlps, attentions)), "?"


def sublayer_table(file, model, dataset):
    runs = pd.read_csv(EXP / "A2_sublayers" / file)
    runs = runs[runs["model"].str.contains(model) & (runs["dataset"] == dataset)]
    runs = runs[runs["head_dict"].fillna("{}").astype(str) == "{}"]  # leave out head-pruning runs
    runs["accuracy"] = pd.to_numeric(runs["accuracy"], errors="coerce")
    runs[["Component", "Approximation", "Translator"]] = runs.apply(what_was_replaced, axis=1, result_type="expand")

    key = ["Component", "Approximation", "Translator", "seed"]
    n_disagree = (runs.groupby(key)["accuracy"].nunique() > 1).sum()
    print(f"{model} / {dataset}: {runs.duplicated(key).sum()} duplicate seed rows dropped, {n_disagree} of them disagreed")
    runs = runs.drop_duplicates(key, keep="last")

    table = []
    for (approximation, component, translator), sel in runs.groupby(["Approximation", "Component", "Translator"]):
        table.append({"Approximation": approximation, "Component": component, "Translator": translator,
                      "acc": f"{sel['accuracy'].mean() * 100:.2f} ({sel['accuracy'].std() * 100:.2f})", "n": len(sel)})
    table = pd.DataFrame(table)
    # one column per translator, like the PDF (Identity | Linear)
    wide = table.pivot(index=["Approximation", "Component"], columns="Translator", values="acc").fillna("–")
    return wide

## Table A.2 — Blocks, MLP or attention sublayers on ImageNet-1k (p. 14)

DINOv2-B and ViT-L reproduce exactly. For DeiT-S, 24 configurations were run twice; this table uses the last run,
and a few cells differ from the PDF by at most 0.08 pp (within one SD).

In [27]:
for file, model in [("results_full_compare.csv", "deit-small"), ("results_full_amended.csv", "dinov2-base"),
                    ("results_full_amended.csv", "vit-large")]:
    display(sublayer_table(file, model, "imagenet-1k"))

deit-small / imagenet-1k: 150 duplicate seed rows dropped, 24 of them disagreed


Translator                   identity        linear
Approximation Component                            
              Original   73.74 (0.14)             –
0->1          Block      64.02 (0.19)  62.52 (0.13)
1             MLP                   –  66.94 (0.11)
10, 11        MLP                   –  71.59 (0.37)
10->11        Block      73.55 (0.21)  73.56 (0.25)
11            Attention  73.48 (0.15)             –
              MLP        73.81 (0.05)  73.49 (0.10)
2->3          Block      70.51 (0.04)  73.30 (0.26)
2->4          Block      63.23 (0.17)  69.75 (0.16)
3             Attention  72.84 (0.09)             –
              MLP        72.04 (0.11)  73.34 (0.10)
3, 4          MLP                   –  70.79 (0.14)
3->4, 9->10   Block      68.86 (0.34)  70.80 (0.18)
3->4, 9->11   Block      66.89 (0.29)  68.22 (0.15)
4, 10         Attention  71.28 (0.20)             –
              MLP        71.15 (0.11)  71.57 (0.15)
4, 10, 11     Attention  69.76 (0.19)             –
              MLP        71.02 (0.11)  70.77 (0.15)
9->11         Block      69.72 (0.12)  69.72 (0.20)

dinov2-base / imagenet-1k: 0 duplicate seed rows dropped, 0 of them disagreed


Translator                      identity        linear
Approximation    Component                            
                 Original   74.18 (0.08)             –
0->1             Block      65.84 (0.19)  73.71 (0.28)
0->1, 2->3       Block      12.65 (0.21)  71.21 (0.63)
0->1, 2->3, 4->5 Block       6.71 (0.09)  61.75 (0.08)
0->4             Block       3.58 (0.27)  27.63 (0.80)
1                Attention  73.02 (0.32)             –
                 MLP        72.00 (0.21)  73.48 (0.07)
1, 2, 3, 4       Attention   9.27 (0.49)             –
                 MLP         6.05 (0.26)  51.95 (0.35)
1, 3             Attention  45.70 (0.09)             –
                 MLP        53.89 (0.16)  71.01 (0.23)
1, 3, 5          Attention  15.45 (0.11)             –
                 MLP        24.33 (0.63)  68.22 (0.23)
5->6             Block      29.38 (0.38)  73.11 (0.58)
6                Attention  55.34 (0.29)             –
                 MLP        18.94 (0.22)  73.15 (0.22)

vit-large / imagenet-1k: 0 duplicate seed rows dropped, 0 of them disagreed


Translator                              identity        linear
Approximation            Component                            
                         Original   78.46 (0.15)             –
16->17, 22->23           Block      77.69 (0.27)  77.74 (0.25)
17, 23                   Attention  77.72 (0.27)             –
                         MLP        78.37 (0.32)  77.76 (0.26)
17->23                   Block      66.60 (0.20)  66.96 (0.09)
18, 19, 20, 21, 22, 23   Attention  66.71 (0.32)             –
                         MLP        72.55 (0.38)  73.37 (0.05)
2->4, 18->23             Block      63.17 (0.15)  67.44 (0.01)
22->23                   Block      78.47 (0.18)  78.12 (0.07)
23                       Attention  78.28 (0.10)             –
                         MLP        78.48 (0.38)  78.13 (0.11)
3, 4, 19, 20, 21, 22, 23 Attention  65.30 (0.06)             –
                         MLP        70.84 (0.36)  72.26 (0.18)
3->4                     Block      77.26 (0.16)  78.45 (0.21)
3->4, 19->23             Block      70.49 (0.22)  71.58 (0.15)
3->4, 20->23             Block      73.55 (0.09)  73.97 (0.37)
3->4, 21->23             Block      75.58 (0.24)  76.20 (0.32)
4                        Attention  77.88 (0.28)             –
                         MLP        77.69 (0.27)  77.86 (0.13)
4, 20, 21, 22, 23        Attention  72.06 (0.14)             –
                         MLP        74.13 (0.16)  74.39 (0.56)
4, 21, 22, 23            Attention  74.14 (0.17)             –
                         MLP        75.79 (0.26)  75.15 (0.11)
4, 22, 23                Attention  76.06 (0.14)             –
                         MLP        77.04 (0.36)  76.38 (0.49)
7->8, 15->16             Block      76.81 (0.06)  77.39 (0.25)
8, 16                    Attention  77.19 (0.27)             –
                         MLP        77.92 (0.13)  77.78 (0.17)

## Table A.3 — Blocks, MLP or attention sublayers on CIFAR-100 (p. 15)

In [28]:
for model in ["deit-small", "dinov2-base", "vit-large"]:
    display(sublayer_table("results_full_table3.csv", model, "cifar100"))

deit-small / cifar100: 21 duplicate seed rows dropped, 0 of them disagreed


Translator                                          ?      identity        linear
Approximation        Component                                                   
                     Original                       –  71.25 (0.04)             –
([(9, 11)], [4], []) mixed (not in PDF)  71.39 (0.15)             –             –
10                   Attention                      –  71.44 (0.21)             –
                     MLP                            –  71.79 (0.33)  71.67 (0.18)
10, 11               Attention                      –  71.65 (0.27)             –
                     MLP                            –  71.91 (0.18)  71.76 (0.19)
3->4, 9->11          Block                          –  68.61 (0.03)  70.95 (0.06)
4                    MLP                            –             –  70.89 (0.32)
4, 10, 11            Attention                      –  69.86 (0.25)             –
                     MLP                            –  70.00 (0.09)  71.14 (0.13)
8->9                 Block                          –  71.86 (0.34)  71.25 (0.27)
9                    Attention                      –  71.20 (0.18)             –
                     MLP                            –  71.97 (0.30)  71.49 (0.20)
9->10                Block                          –  71.93 (0.11)  71.88 (0.38)
9->11                Block                          –  72.50 (0.13)  72.09 (0.33)

dinov2-base / cifar100: 21 duplicate seed rows dropped, 0 of them disagreed


Translator                                         ?      identity        linear
Approximation       Component                                                   
                    Original                       –  87.21 (0.47)             –
([(2, 3)], [], [1]) mixed (not in PDF)  65.96 (0.46)             –             –
0->1                Block                          –  79.50 (0.40)  86.05 (0.24)
0->1, 2->3          Block                          –  33.32 (0.86)  84.07 (0.50)
0->1, 2->3, 4->5    Block                          –  29.31 (1.55)  78.78 (0.32)
0->4                Block                          –  17.77 (1.09)  61.75 (0.54)
1                   Attention                      –  83.90 (0.24)             –
                    MLP                            –  81.18 (0.22)  86.07 (0.35)
1, 2, 3, 4          Attention                      –  27.80 (0.50)             –
                    MLP                            –  30.49 (1.73)  68.75 (0.05)
1, 3                Attention                      –  43.01 (1.38)             –
                    MLP                            –  51.96 (1.03)  83.60 (0.48)
1, 3, 5             Attention                      –  35.05 (0.92)             –
                    MLP                            –  38.25 (1.26)  81.47 (0.34)
2->3                Block                          –  47.96 (1.29)  86.10 (0.21)
3                   Attention                      –  56.50 (0.53)             –
                    MLP                            –  60.13 (0.68)  86.04 (0.35)

vit-large / cifar100: 18 duplicate seed rows dropped, 0 of them disagreed


Translator                                                  ?      identity        linear
Approximation                Component                                                   
                             Original                       –  86.37 (0.25)             –
([(3, 4)], [21, 22, 23], []) mixed (not in PDF)  86.77 (0.28)             –             –
17->23                       Block                          –  85.53 (0.46)  85.08 (0.62)
18, 19, 20, 21, 22, 23       Attention                      –  83.69 (0.64)             –
                             MLP                            –  86.06 (0.28)  86.14 (0.29)
2->4, 18->23                 Block                          –  74.12 (0.08)  84.03 (0.06)
20->21                       Block                          –  86.47 (0.14)  86.45 (0.31)
20->22                       Block                          –  86.46 (0.11)  86.22 (0.16)
20->23                       Block                          –  86.32 (0.06)  86.36 (0.47)
21                           Attention                      –  86.43 (0.26)             –
                             MLP                            –  86.54 (0.11)  86.43 (0.19)
21, 22                       Attention                      –  86.08 (0.16)             –
                             MLP                            –  86.40 (0.42)  86.48 (0.08)
21, 22, 23                   Attention                      –  85.79 (0.17)             –
                             MLP                            –  86.75 (0.21)  86.47 (0.14)
21->22                       Block                          –  86.55 (0.09)  86.52 (0.18)
22                           Attention                      –  86.26 (0.24)             –
                             MLP                            –  86.61 (0.17)  86.51 (0.09)
3, 4, 19, 20, 21, 22, 23     Attention                      –  72.92 (0.47)             –
                             MLP                            –  82.46 (0.11)  84.82 (0.27)
3->4                         Block                          –             –  86.11 (0.29)
3->4, 19->23                 Block                          –  84.11 (0.21)  85.79 (0.57)
3->4, 20->23                 Block                          –  84.57 (0.33)  86.31 (0.44)
3->4, 21->22                 Block                          –  84.60 (0.11)  86.28 (0.22)
3->4, 21->23                 Block                          –  84.65 (0.02)  86.43 (0.26)
4, 20, 21, 22, 23            Attention                      –  83.83 (0.20)             –
                             MLP                            –  85.12 (0.25)  85.98 (0.16)
4, 21, 22, 23                Attention                      –  84.10 (0.08)             –
                             MLP                            –  85.38 (0.25)  86.25 (0.46)
4, 22                        Attention                      –  84.73 (0.13)             –
                             MLP                            –  85.24 (0.17)  86.24 (0.30)
4, 22, 23                    Attention                      –  84.53 (0.04)             –
                             MLP                            –  85.49 (0.30)  86.31 (0.43)

---
# Appendix B: models and datasets (Tables B.1, B.2)

## Table B.1 — Pretrained models (p. 16)

Mostly static. This checks that the Hugging Face keys and block counts match what was run (d is checked in Table A.1).

In [29]:
table = []
for model, dataset in NATURAL + MEDICAL:
    if (model, dataset) in NATURAL:
        accuracy_file = S2 / "accuracies" / f"accuracies_results_{model}_{dataset}.csv"
    else:
        accuracy_file = S4 / "accuracies" / "combined" / f"accuracies_results_{model}_{dataset}.csv"
    table.append({"Alias": MODEL_NAME[model], "Hugging Face key": pd.read_csv(accuracy_file)["model"].iloc[0],
                  "Blocks (last span end + 1)": load_metrics(model, dataset)["end"].max() + 1})
pd.DataFrame(table).drop_duplicates()

,Alias,Hugging Face key,Blocks (last span end + 1)
0,DeiT-S,facebook/deit-small-patch16-224,12
2,ViT-S,WinKawaks/vit-small-patch16-224,12
4,DINOv2-B,facebook/dinov2-base,12
6,ViT-L,google/vit-large-patch16-224,24
8,RAD-DINO,microsoft/rad-dino,12
10,RAD-DINO-MAIRA-2,microsoft/rad-dino-maira-2,12
12,Phikon-v2,owkin/phikon-v2,24


## Table B.2 — Datasets (p. 16)

The result files only store the **test** majority-class rate (`majority_class_rate`) and the number of calibration
images (`num_samples`, should be 500 as in Sec. 3). Class counts and split sizes are not in them.

In [30]:
files = list((S2 / "results").glob("*.csv")) + list((S4 / "results" / "combined").glob("*.csv"))
results = pd.concat([pd.read_csv(p, usecols=["dataset", "majority_class_rate", "num_samples"]) for p in files])
table = []
for dataset, sel in results.groupby("dataset"):
    table.append({"dataset": dataset,
                  "majority-class rate (test)": sorted(sel["majority_class_rate"].dropna().round(4).unique()),
                  "calibration images": sorted(sel["num_samples"].unique())})
pd.DataFrame(table)

,dataset,majority-class rate (test),calibration images
0,chestxray14,[0.535],[500]
1,chestxray14_multilabel,[],[500]
2,chestxray_pneumonia,[0.625],[500]
3,cifar100,[0.01],[500]
4,imagenet-1k,[0.0019],[500]
5,patchcamelyon,[0.5002],[500]
6,tcga_ut,[0.0871],[500]
